# Indonesia Captive RE Analysis (Solar + Wind)

## Setup

In [ ]:
# TSCC Cluster ONLY: use the lab storage space to avoid re-downloading large datasets.
# RUN THIS BEFORE ANY OTHER CELL.
%setenv GEODATA_ROOT /tscc/projects/ps-davidson/geodata

In [ ]:
import geodata
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import Point

## Configuration

In [ ]:
# --- Spatial / temporal extent (Indonesia) ---
YEAR        = 2018
XS          = slice(95, 141)   # longitude
YS          = slice(-11, 6)    # latitude

# --- Technology (names must exist in geodata's resource library) ---
SOLAR_PANEL = "KANEKA"                  # geodata.pv panel
SOLAR_ORIENT= "latitude_optimal"
WIND_TURBINE= "Siemens_SWT_107_3600kW"  # ~3.6 MW, 90 m hub height

# --- Siting constraints ---
BUFFER_KM        = 10        # captive buffer radius around each facility
SLOPE_MAX_SOLAR  = 5         # %, exclude steeper cells for solar
SLOPE_MAX_WIND   = 20        # %, exclude steeper cells for wind

# Land packing factors [MW/km^2] (from the siting README; tune in sensitivities)
PACKING_SOLAR = 30.0
PACKING_WIND  = 3.9

# --- Input data (edit to your paths) ---
LANDUSE_TIF  = "data/KLHK_landuse_indonesia.tif"   # KLHK land-cover raster
SLOPE_TIF    = "data/USGS_slope_indonesia.tif"     # USGS/EarthExplorer slope raster (percent)
FACILITY_CSV = "data/industrial_facilities.csv"    # cols: Facility, Latitude, Longitude

# KLHK land-cover class codes to KEEP as developable (exclude forest/residential/water).
# TODO: set to the integer codes for your KLHK product (placeholder values shown).
SUITABLE_LANDUSE_CLASSES = [20, 21, 2010, 2014, 2500]

## Download Indonesia ERA5 Weather Data

In [ ]:
dataset = geodata.Dataset(
    module="era5",
    weather_data_config="wind_solar_hourly",
    years=slice(YEAR, YEAR),
    months=slice(1, 12),
)

In [ ]:
# Download only if not already staged locally
if not dataset.prepared:
    dataset.get_data()

## Create Cutout

In [ ]:
cutout = geodata.Cutout(
    name=f"indonesia-{YEAR}-solar-wind",
    module="era5",
    weather_data_config="wind_solar_hourly",
    xs=XS,
    ys=YS,
    years=slice(YEAR, YEAR),
    months=slice(1, 12),
)
cutout

In [ ]:
cutout.prepare()

## Solar Capacity Factor

In [ ]:
solar_ts = (
    cutout.pv(panel=SOLAR_PANEL, orientation=SOLAR_ORIENT)
    .reset_coords(["lon", "lat"], drop=True)
    .rename({"x": "lon", "y": "lat"})
)

# Annual mean CF (one value per grid cell)
solar_cf = solar_ts.mean(dim="time").to_dataset(name="solar_cf")
solar_cf

## Wind Capacity Factor

In [ ]:
wind_ts = (
    cutout.wind(turbine=WIND_TURBINE)
    .reset_coords(["lon", "lat"], drop=True)
    .rename({"x": "lon", "y": "lat"})
)

wind_cf = wind_ts.mean(dim="time").to_dataset(name="wind_cf")
wind_cf

## Plot Capacity Factors

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
solar_cf["solar_cf"].plot(ax=axes[0], cmap="YlOrRd")
axes[0].set_title(f"Solar CF — {YEAR}")
wind_cf["wind_cf"].plot(ax=axes[1], cmap="GnBu")
axes[1].set_title(f"Wind CF — {YEAR}")
plt.tight_layout()

## Build Suitability Mask

In [ ]:
mask = geodata.Mask(
    "indonesia_captive",
    layer_path=[LANDUSE_TIF, SLOPE_TIF],
    layer_name=["landuse", "slope"],
)
mask

In [ ]:
# Land use -> 1 where the class is developable, else 0
mask.filter_layer(
    "landuse",
    dest_layer_name="landuse_ok",
    values=SUITABLE_LANDUSE_CLASSES,
    binarize=True,
)

# Slope -> 1 where slope is below each technology threshold, else 0
mask.filter_layer("slope", dest_layer_name="slope_solar", max_bound=SLOPE_MAX_SOLAR, binarize=True)
mask.filter_layer("slope", dest_layer_name="slope_wind",  max_bound=SLOPE_MAX_WIND,  binarize=True)

In [ ]:
# Combine constraints (AND): suitable only where land use AND slope both pass
solar_suitable = mask.merge_layer(
    method="and", layers=["landuse_ok", "slope_solar"], show_raster=True
)
# merged_mask now holds the solar suitability surface
mask

## Industrial Facilities + 10-km Buffers

In [ ]:
facilities = pd.read_csv(FACILITY_CSV)
facilities = facilities[["Facility", "Latitude", "Longitude"]].dropna().drop_duplicates()

gdf = gpd.GeoDataFrame(
    facilities,
    geometry=[Point(xy) for xy in zip(facilities["Longitude"], facilities["Latitude"])],
    crs="EPSG:4326",
)

# Buffer in meters (EPSG:6933 is an equal-area metric CRS), then back to lat/lon
gdf["buffer"] = gdf.to_crs("EPSG:6933").buffer(BUFFER_KM * 1000).to_crs("EPSG:4326")

# {facility_name: buffered_polygon} for extract_shapes
buffers = dict(zip(gdf["Facility"], gdf["buffer"]))
gdf.head()

## Extract Suitable Area within Each Buffer

In [ ]:
solar_shapes = mask.extract_shapes(buffers, show_raster=False)
list(solar_shapes.keys())[:5]

## Deployable Potential per Facility

In [ ]:
from geodata import mask as gmask

def cell_area_km2(da):
    """Approximate per-cell area [km^2] from lat/lon spacing (cos-lat correction)."""
    dlon = float(abs(da.lon[1] - da.lon[0]))
    dlat = float(abs(da.lat[1] - da.lat[0]))
    lat = np.deg2rad(da.lat.values)
    km_per_deg = 111.32
    aw = dlon * km_per_deg * np.cos(lat)          # width [km] per row
    ah = dlat * km_per_deg                        # height [km]
    return xr.DataArray(np.outer(aw * ah, np.ones(da.lon.size)),
                        coords={"lat": da.lat, "lon": da.lon}, dims=["lat", "lon"])

def facility_potential(name, shape_raster, cf_ds, cf_var, packing):
    """Suitable area, mean CF, and deployable capacity within one facility buffer."""
    suit = gmask.ras_to_xarr(shape_raster)        # suitable mask (0/1) as DataArray
    cf   = cf_ds[cf_var].reindex_like(suit, method="nearest")
    area = cell_area_km2(suit) * (suit > 0)
    suit_area = float(area.sum())
    mean_cf   = float(cf.where(suit > 0).mean())
    return {
        "Facility": name,
        "suitable_area_km2": suit_area,
        "mean_CF": mean_cf,
        "capacity_MW": suit_area * packing,
    }

def batch_potential(shapes, cf_ds, cf_var, packing):
    rows = [facility_potential(n, r, cf_ds, cf_var, packing) for n, r in shapes.items()]
    return pd.DataFrame(rows).sort_values("mean_CF", ascending=False)

In [ ]:
solar_potential = batch_potential(solar_shapes, solar_cf, "solar_cf", PACKING_SOLAR)
solar_potential.head(10)

### Wind potential

In [ ]:
wind_suitable = mask.merge_layer(
    method="and", layers=["landuse_ok", "slope_wind"], show_raster=False
)
wind_shapes = mask.extract_shapes(buffers, show_raster=False)
wind_potential = batch_potential(wind_shapes, wind_cf, "wind_cf", PACKING_WIND)
wind_potential.head(10)

## Results

In [ ]:
results = (
    solar_potential.rename(columns={"mean_CF": "solar_CF", "capacity_MW": "solar_MW"})
    .merge(
        wind_potential.rename(columns={"mean_CF": "wind_CF", "capacity_MW": "wind_MW"})
                      [["Facility", "wind_CF", "wind_MW"]],
        on="Facility", how="outer",
    )
)
results.to_csv(f"captive_potential_{YEAR}.csv", index=False)
results.head(10)